# 1. Install Dependencies

In [1]:
# Kaggle's image already ships torch, transformers, peft, accelerate and huggingface-hub.
# Only bitsandbytes is missing, and we deliberately do NOT pin or reinstall the others
# (a downgrade could break the preinstalled torch build).
!pip install -q bitsandbytes

import accelerate
import bitsandbytes
import peft
import torch
import transformers

print("bitsandbytes", bitsandbytes.__version__)
print(
    "transformers",
    transformers.__version__,
    "| peft",
    peft.__version__,
    "| accelerate",
    accelerate.__version__,
    "| torch",
    torch.__version__,
)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.6 MB/s eta 0:00:00:00:0100:01
bitsandbytes 0.50.2
transformers 5.0.0 | peft 0.19.1 | accelerate 1.13.0 | torch 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


# 2. HuggingFace Authentication

In [2]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("Logged in as:", whoami()["name"])

Logged in as: muhasin-code


# 3: Write the Evaluation Module

In [3]:
%%writefile evaluate_extraction.py
"""Batched evaluation for the PaperLens extraction adapter (QLoRA on Qwen2.5-3B-Instruct)."""
from __future__ import annotations

import gc
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from tqdm.auto import tqdm

BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_REPO = "muhasin-code/paperlens-qwen2.5-3b-extraction-v3"

FIELDS = ["research_question", "method", "datasets", "metrics", "key_finding"]
LIST_FIELDS = {"datasets", "metrics"}

# Fallback only: used if a val row has no system message.
# Must match src/paperlens/extraction/annotator.py.
DEFAULT_SYSTEM_PROMPT = """You are a research assistant specialized in extracting structured information from machine learning papers.

From the provided paper chunk, extract the following fields and return ONLY a JSON object with these keys (all nullable):

- research_question: The core problem or hypothesis the paper addresses
- method: The proposed approach or algorithm (max one paragraph)
- datasets: List of dataset names used (empty array if none)
- metrics: List of evaluation metrics reported (empty array if none)
- key_finding: The main result or contribution in one sentence

If a field is not present or not useful, set it to null. Do not hallucinate information. Do not add any prose - only return the JSON object.

Example output:
{"research_question": "How to improve...", "method": "...", "datasets": ["..."], "metrics": ["..."], "key_finding": "..."}"""


# ----------------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------------
def find_val_file(root: str = "/kaggle/input") -> Path:
    """Locate val.jsonl wherever the Kaggle dataset got mounted."""
    matches = list(Path(root).rglob("val.jsonl"))
    if not matches:
        raise FileNotFoundError(f"No val.jsonl found under {root}. Is the dataset attached?")
    return matches[0]


def load_jsonl(path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_model(base_id: str = BASE_MODEL_ID, adapter_id: str = ADAPTER_REPO, token: str | None = None):
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(base_id, padding_side="left")  # left-pad for batched generation
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,  # T4: no bf16
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        base_id,
        quantization_config=bnb,
        device_map={"": 0},          # pin to ONE GPU; "auto" can split across 2x T4 and slow generation
        dtype=torch.float16,         # transformers>=5 name (older versions: torch_dtype)
        attn_implementation="sdpa",  # FlashAttention-2 is not supported on T4 (Turing)
    )
    model = PeftModel.from_pretrained(model, adapter_id, token=token)
    model.eval()
    return tok, model


# ----------------------------------------------------------------------------
# Data handling
# ----------------------------------------------------------------------------
def parse_json(text: str | None) -> dict | None:
    if not text:
        return None
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def get_truth(ex: dict) -> dict | None:
    """Ground truth from `metadata` if present, else from a trailing assistant message."""
    meta = ex.get("metadata") or {}
    if any(f in meta for f in FIELDS):
        return {f: meta.get(f) for f in FIELDS}
    msgs = ex.get("messages") or []
    if msgs and msgs[-1].get("role") == "assistant":
        parsed = parse_json(msgs[-1].get("content"))
        if parsed:
            return {f: parsed.get(f) for f in FIELDS}
    return None


def split_messages(ex: dict) -> tuple[str | None, str]:
    msgs = ex["messages"]
    system = next((m["content"] for m in msgs if m.get("role") == "system"), None)
    user = next(m["content"] for m in msgs if m.get("role") == "user")
    return system, user


def build_prompt(tok, system: str, user: str, mode: str, max_user_tokens: int) -> str:
    # Truncate the *user text* (not the finished prompt) so the generation prompt is never cut off.
    ids = tok(user, add_special_tokens=False)["input_ids"]
    if len(ids) > max_user_tokens:
        user = tok.decode(ids[:max_user_tokens])
    if mode == "chat":
        return tok.apply_chat_template(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )
    if mode == "raw":
        return f"{system}\n\n{user}"
    raise ValueError(f"Unknown mode: {mode!r}")


# ----------------------------------------------------------------------------
# Generation (batched, length-sorted, OOM back-off)
# ----------------------------------------------------------------------------
@torch.inference_mode()
def generate_batched(tok, model, prompts: list[str], batch_size: int = 8, max_new_tokens: int = 256) -> list[str]:
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))  # similar lengths -> less padding
    outputs: list[str | None] = [None] * len(prompts)
    bar = tqdm(total=len(prompts), desc="Generating")
    pos = 0
    while pos < len(order):
        idx = order[pos : pos + batch_size]
        try:
            enc = tok([prompts[j] for j in idx], return_tensors="pt", padding=True).to(model.device)
            gen = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.0,  # explicit, so the model's generation_config can't change results
                pad_token_id=tok.pad_token_id,
            )
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print(f"OOM -> batch_size={batch_size}")
            continue
        new_tokens = gen[:, enc["input_ids"].shape[1]:]  # decode ONLY newly generated tokens
        for j, text in zip(idx, tok.batch_decode(new_tokens, skip_special_tokens=True)):
            outputs[j] = text
        pos += len(idx)
        bar.update(len(idx))
    bar.close()
    return outputs  # type: ignore[return-value]


# ----------------------------------------------------------------------------
# Scoring
# ----------------------------------------------------------------------------
def _as_set(field: str, v) -> set[str]:
    if field in LIST_FIELDS:
        items = v if isinstance(v, list) else [v]
        return {str(x).strip().lower() for x in items}
    return set(str(v).lower().split())


def _prf(pred: set, truth: set) -> tuple[float, float, float]:
    if not pred and not truth:
        return 1.0, 1.0, 1.0
    tp = len(pred & truth)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(truth) if truth else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f


def score_field(field: str, pred, truth, empty_is_null: bool = True):
    """Returns ((precision, recall, f1), exact_match)."""
    if empty_is_null and field in LIST_FIELDS:
        pred = pred or None
        truth = truth or None
    if pred is None and truth is None:
        return (1.0, 1.0, 1.0), True
    if pred is None or truth is None:
        return (0.0, 0.0, 0.0), False
    a, b = _as_set(field, pred), _as_set(field, truth)
    return _prf(a, b), a == b if field in LIST_FIELDS else str(pred).strip().lower() == str(truth).strip().lower()


# ----------------------------------------------------------------------------
# Evaluation
# ----------------------------------------------------------------------------
def evaluate(
    tok,
    model,
    data: list[dict],
    mode: str = "chat",
    batch_size: int = 8,
    max_new_tokens: int = 256,
    max_user_tokens: int = 900,
    limit: int | None = None,
    empty_is_null: bool = True,
    adapter_repo: str | None = ADAPTER_REPO,
    label: str = "model",
) -> dict:
    rows = data[:limit] if limit else data

    prompts, truths = [], []
    for i, ex in enumerate(rows):
        truth = get_truth(ex)
        if truth is None:
            raise ValueError(f"Row {i}: no ground truth found. Keys present: {list(ex.keys())}")
        system, user = split_messages(ex)
        prompts.append(build_prompt(tok, system or DEFAULT_SYSTEM_PROMPT, user, mode, max_user_tokens))
        truths.append(truth)

    t0 = time.time()
    raw = generate_batched(tok, model, prompts, batch_size, max_new_tokens)
    elapsed = time.time() - t0

    acc = {f: {"precision": [], "recall": [], "f1": [], "exact": []} for f in FIELDS}
    details, valid = [], 0
    for i, (text, truth) in enumerate(zip(raw, truths)):
        pred = parse_json(text)
        is_valid = pred is not None and all(f in pred for f in FIELDS)
        valid += is_valid
        for f in FIELDS:
            if pred is None:  # unparseable output scores zero everywhere
                (p, r, f1), ex_ok = (0.0, 0.0, 0.0), False
            else:
                (p, r, f1), ex_ok = score_field(f, pred.get(f), truth[f], empty_is_null)
            for k, v in zip(("precision", "recall", "f1", "exact"), (p, r, f1, ex_ok)):
                acc[f][k].append(float(v))
        details.append({"index": i, "valid_json": is_valid, "raw_output": text, "predicted": pred, "ground_truth": truth})

    n = len(rows)
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    summary = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "base_model": BASE_MODEL_ID,
        "adapter_repo": adapter_repo,
        "label": label,
        "prompt_mode": mode,
        "total_examples": n,
        "json_valid_rate": valid / n if n else 0.0,
        "seconds": round(elapsed, 1),
        "sec_per_example": round(elapsed / n, 2) if n else None,
        "fields": {f: {k: mean(v) for k, v in acc[f].items()} for f in FIELDS},
        "scoring": "macro-averaged per-example set-F1 (token set for text fields, item set for list fields)",
        "empty_is_null": empty_is_null,
    }
    return {"summary": summary, "examples": details}


def probe_prompt_modes(tok, model, data, n: int = 24, batch_size: int = 8) -> str:
    """Quick A/B on n examples to pick between chat-template and raw prompts."""
    scores = {}
    for mode in ("chat", "raw"):
        s = evaluate(tok, model, data, mode=mode, limit=n, batch_size=batch_size)["summary"]
        mean_f1 = sum(v["f1"] for v in s["fields"].values()) / len(FIELDS)
        scores[mode] = (s["json_valid_rate"], mean_f1)
        print(f"{mode:>4}: valid={s['json_valid_rate']:.0%}  mean F1={mean_f1:.3f}")
    best = max(scores, key=lambda m: scores[m])
    print(f"-> using '{best}'")
    return best


# ----------------------------------------------------------------------------
# Reporting
# ----------------------------------------------------------------------------
def print_report(res: dict) -> None:
    s = res["summary"]
    print("=" * 64)
    print(f"EVALUATION [{s.get('label', 'model')}]  ({s['total_examples']} examples, prompt_mode={s['prompt_mode']})")
    print(f"JSON validity: {s['json_valid_rate']:.1%}   time: {s['seconds']}s ({s['sec_per_example']} s/ex)")
    print("-" * 64)
    print(f"{'field':20s} {'P':>6} {'R':>6} {'F1':>6} {'exact':>7}")
    for f, v in s["fields"].items():
        print(f"{f:20s} {v['precision']:6.3f} {v['recall']:6.3f} {v['f1']:6.3f} {v['exact']:7.1%}")


def show_diffs(res: dict, field: str, n: int = 5) -> None:
    """Side-by-side predicted vs ground truth for a field (for diagnosing paraphrase vs real errors)."""
    shown = 0
    for ex in res["examples"]:
        if ex["predicted"] is None:
            continue
        p, t = ex["predicted"].get(field), ex["ground_truth"][field]
        if str(p).strip().lower() != str(t).strip().lower():
            print(f"[{ex['index']}] {field}\n  PRED : {p}\n  TRUTH: {t}\n")
            shown += 1
            if shown >= n:
                break


def save_results(res: dict, path: str = "/kaggle/working/extraction_eval_results.json") -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2, ensure_ascii=False)
    print(f"Saved -> {path}")

Writing evaluate_extraction.py


# 4: Load Base Model + v3 Adapter

In [ ]:
import gc
import json
import re
import statistics
import time
from collections import Counter
from pathlib import Path
from statistics import mean

import evaluate_extraction as ee
import torch
from evaluate_extraction import *  # noqa: F403, F405

print("Adapter:", ee.ADAPTER_REPO)
assert ee.ADAPTER_REPO.endswith("extraction-v3"), "module still points at an old adapter"

tok, model = load_model(token=HF_TOKEN)

gc.collect()
torch.cuda.empty_cache()
print("Adapters loaded:", list(model.peft_config.keys()))
print("Model device:", model.device)
print(
    f"GPU memory: allocated={torch.cuda.memory_allocated() / 1e9:.2f} GB, "
    f"reserved={torch.cuda.memory_reserved() / 1e9:.2f} GB"
)
print("Tokenizer padding side:", tok.padding_side, "| pad token:", tok.pad_token)

Adapter: muhasin-code/paperlens-qwen2.5-3b-extraction-v3


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/120M [00:00<?, ?B/s]

Adapters loaded: ['default']
Model device: cuda:0
GPU memory: allocated=2.18 GB, reserved=2.37 GB
Tokenizer padding side: left | pad token: <|endoftext|>


# 5: Load Data, Verify the Paper-Level Split, Locate the Gold File

In [5]:
val_path = find_val_file()
val_data = load_jsonl(val_path)
train_data = load_jsonl(val_path.parent / "train.jsonl")
print(f"val file: {val_path}")
print(f"train={len(train_data)} val={len(val_data)}  (expected 2082 / 233)")


def is_null(v):
    return v is None or v == [] or v == ""


# ---- ground truth, label distribution, input length ----
tr_t = [get_truth(e) for e in train_data]
va_t = [get_truth(e) for e in val_data]
assert all(tr_t) and all(va_t), "some rows have no ground truth"
print(f"\n{'non-null rate':20s} {'train':>7s} {'val':>7s}")
for f in FIELDS:
    a = sum(not is_null(t[f]) for t in tr_t) / len(tr_t)
    b = sum(not is_null(t[f]) for t in va_t) / len(va_t)
    print(f"{f:20s} {a:7.1%} {b:7.1%}")
lens = [len(tok(split_messages(e)[1], add_special_tokens=False)["input_ids"]) for e in val_data]
print(
    f"\nuser-text tokens: median={int(statistics.median(lens))}, max={max(lens)}, over 900: {sum(l > 900 for l in lens)}"
)

# ---- paper-level split ----
tr_ids = {e["metadata"]["arxiv_id"] for e in train_data}
va_ids = {e["metadata"]["arxiv_id"] for e in val_data}
tr_chunks = {e["metadata"]["chunk_id"] for e in train_data}
va_chunks = {e["metadata"]["chunk_id"]: i for i, e in enumerate(val_data)}
print(f"\npapers: train={len(tr_ids)} val={len(va_ids)} overlap={len(tr_ids & va_ids)}")
print(f"chunk_ids in both train and val: {len(tr_chunks & set(va_chunks))}")

# ---- gold file: locate and describe (no assumptions about where its chunks live) ----
gold = None
gold_files = list(Path("/kaggle/input").rglob("gold_extraction.jsonl"))
print("\nGold file(s):", [str(p) for p in gold_files])
if gold_files:
    gold_rows = load_jsonl(gold_files[0])
    print(f"gold rows: {len(gold_rows)} | keys in row 0: {list(gold_rows[0].keys())}")
    gold = {r["chunk_id"]: r for r in gold_rows}
    g_ids = list(gold)
    print(
        f"gold chunk_ids in val: {sum(i in va_chunks for i in g_ids)} | in train: {sum(i in tr_chunks for i in g_ids)} | "
        f"in neither: {sum(i not in va_chunks and i not in tr_chunks for i in g_ids)}"
    )
    # paper-level check, assuming chunk_id looks like "<arxiv_id>_<section>"
    g_papers = {i.rsplit("_", 1)[0] for i in g_ids}
    print(
        f"gold papers also in TRAIN papers: {len(g_papers & tr_ids)} of {len(g_papers)} (assumes '<arxiv_id>_<section>' ids)"
    )
    print(
        f"non-empty gold_method: {sum(bool(r.get('gold_method')) for r in gold_rows)} | "
        f"gold_key_finding: {sum(bool(r.get('gold_key_finding')) for r in gold_rows)}"
    )
else:
    print(
        "-> Not found. Attach a dataset containing gold_extraction.jsonl (Add Input), or the gold section will be skipped."
    )

assert not (tr_ids & va_ids), "train and val still share papers"

val file: /kaggle/input/datasets/muhammedmuhasink/paperlens-extraction-dataset/val.jsonl
train=2082 val=233  (expected 2082 / 233)

non-null rate          train     val
research_question     100.0%  100.0%
method                 24.0%   23.2%
datasets               24.2%   24.9%
metrics                36.9%   41.6%
key_finding            40.9%   39.9%

user-text tokens: median=595, max=890, over 900: 0

papers: train=781 val=86 overlap=0
chunk_ids in both train and val: 0

Gold file(s): ['/kaggle/input/datasets/muhammedmuhasink/paperlens-extraction-dataset/gold_extraction.jsonl']
gold rows: 20 | keys in row 0: ['chunk_id', 'gold_method', 'gold_key_finding']
gold chunk_ids in val: 20 | in train: 0 | in neither: 0
gold papers also in TRAIN papers: 0 of 19 (assumes '<arxiv_id>_<section>' ids)
non-empty gold_method: 5 | gold_key_finding: 10


# 6: Probe the Prompt Format and Peek at Raw Output

In [6]:
t0 = time.time()
mode = probe_prompt_modes(tok, model, val_data, n=24, batch_size=8)
print(f"Probe took {time.time() - t0:.0f}s (includes CUDA warm-up)")

r = evaluate(tok, model, val_data, mode=mode, limit=4, batch_size=4)
ex0 = r["examples"][0]
print("\nRAW OUTPUT:\n", ex0["raw_output"])
print("\nGROUND TRUTH:\n", json.dumps(ex0["ground_truth"], indent=2))

Generating:   0%|          | 0/24 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


chat: valid=100%  mean F1=0.914


Generating:   0%|          | 0/24 [00:00<?, ?it/s]

 raw: valid=33%  mean F1=0.293
-> using 'chat'
Probe took 190s (includes CUDA warm-up)


Generating:   0%|          | 0/4 [00:00<?, ?it/s]


RAW OUTPUT:
 {"research_question": "What method does the paper 'Spoken Language Identification using ConvNets' propose? Context: Language Identification (LI) is an important first step in several speech processing systems", "method": "was able to achieve an accuracy of 91.2% on dataset comprising of 3 languages \u2013 English, French and German.', 'In Table TABREF1, we summarize the quantitative results of the above previous studies. It includes the model basis, feature description, languages classified and the used dataset along with accuracy obtained. The table also lists the overall results of our proposed models (at the top). The languages used by various authors along with their acronyms are English (En), Spanish (Es), French (Fr", "datasets": null, "metrics": ["accuracy"], "key_finding": null}

GROUND TRUTH:
 {
  "research_question": "What method does the paper 'Spoken Language Identification using ConvNets' propose? Context: Language Identification (LI) is an important first st

# 7: Dry Run on 50 Examples

In [7]:
t0 = time.time()
dry = evaluate(tok, model, val_data, mode="chat", limit=50, batch_size=16, label="v3-dry")
print(f"Wall time: {time.time() - t0:.0f}s")
print_report(dry)

bad = [e for e in dry["examples"] if not e["valid_json"]]
print(f"\nInvalid outputs: {len(bad)}")
for e in bad[:5]:
    n_tok = len(tok(e["raw_output"], add_special_tokens=False)["input_ids"])
    print(f"[{e['index']}] ~{n_tok} new tokens (cap 256)\n...{e['raw_output'][-300:]}\n")

Generating:   0%|          | 0/50 [00:00<?, ?it/s]

Wall time: 132s
EVALUATION [v3-dry]  (50 examples, prompt_mode=chat)
JSON validity: 100.0%   time: 132.3s (2.65 s/ex)
----------------------------------------------------------------
field                     P      R     F1   exact
research_question     1.000  1.000  1.000  100.0%
method                0.991  0.997  0.994   74.0%
datasets              0.900  0.900  0.900   90.0%
metrics               0.947  0.923  0.929   86.0%
key_finding           0.745  0.748  0.738   68.0%

Invalid outputs: 0


# 8: Full Evaluation: Fine-tuned v3

In [8]:
t0 = time.time()
res = evaluate(tok, model, val_data, mode="chat", batch_size=16, label="v3")
print(f"Wall time: {time.time() - t0:.0f}s")
print_report(res)
save_results(res, "/kaggle/working/extraction_eval_v3.json")

parsed_all = [e for e in res["examples"] if e["predicted"] is not None]
print(f"\nParsed: {len(parsed_all)}/{len(res['examples'])}")
print(f"\n{'field':20s} {'truth null':>11s} {'pred null':>10s}   (parsed examples)")
for f in FIELDS:
    t_null = sum(is_null(e["ground_truth"][f]) for e in parsed_all) / len(parsed_all)
    p_null = sum(is_null(e["predicted"].get(f)) for e in parsed_all) / len(parsed_all)
    print(f"{f:20s} {t_null:11.0%} {p_null:10.0%}")

Generating:   0%|          | 0/233 [00:00<?, ?it/s]

Wall time: 516s
EVALUATION [v3]  (233 examples, prompt_mode=chat)
JSON validity: 99.6%   time: 515.5s (2.21 s/ex)
----------------------------------------------------------------
field                     P      R     F1   exact
research_question     0.996  0.996  0.996   99.6%
method                0.990  0.994  0.992   82.0%
datasets              0.951  0.940  0.943   92.3%
metrics               0.974  0.958  0.963   91.4%
key_finding           0.697  0.686  0.686   64.4%
Saved -> /kaggle/working/extraction_eval_v3.json

Parsed: 232/233

field                 truth null  pred null   (parsed examples)
research_question             0%         0%
method                       77%        77%
datasets                     75%        79%
metrics                      58%        59%
key_finding                  60%        54%


# 9: Base Model (Adapter Disabled)

In [9]:
t0 = time.time()
with model.disable_adapter():
    base_res = evaluate(
        tok, model, val_data, mode="chat", batch_size=16, adapter_repo=None, label="base"
    )
print(f"Wall time: {time.time() - t0:.0f}s")
print_report(base_res)
save_results(base_res, "/kaggle/working/extraction_eval_base.json")

# sanity check: the adapter must really have been off for this run
same = mean(
    a["raw_output"] == b["raw_output"]
    for a, b in zip(res["examples"], base_res["examples"], strict=False)
)
print(
    f"\nBase outputs identical to v3 outputs: {same:.1%}  (should be near 0%; ~100% means the adapter was still active)"
)

Generating:   0%|          | 0/233 [00:00<?, ?it/s]

Wall time: 481s
EVALUATION [base]  (233 examples, prompt_mode=chat)
JSON validity: 95.3%   time: 480.7s (2.06 s/ex)
----------------------------------------------------------------
field                     P      R     F1   exact
research_question     0.486  0.218  0.294    0.0%
method                0.090  0.070  0.077    0.0%
datasets              0.202  0.210  0.205   19.7%
metrics               0.327  0.333  0.325   26.6%
key_finding           0.081  0.099  0.084    0.4%
Saved -> /kaggle/working/extraction_eval_base.json

Base outputs identical to v3 outputs: 0.0%  (should be near 0%; ~100% means the adapter was still active)


# 10: Baselines: All-Null and Dictionary Lookup (computed on this train/val split)

In [10]:
def lex_predict(vocab, chunk, min_count=2):
    u = chunk.lower()
    hits = [
        v
        for v, c in vocab.items()
        if c >= min_count and re.search(rf"(?<!\w){re.escape(v.lower())}(?!\w)", u)
    ]
    return hits or None


allnull = {
    f: mean(score_field(f, None, e["ground_truth"][f])[0][2] for e in res["examples"])
    for f in FIELDS
}
lookup = {}
for f in ("metrics", "datasets"):
    vocab = Counter(x for t in tr_t for x in (t[f] or []))
    lookup[f] = mean(
        score_field(f, lex_predict(vocab, split_messages(e)[1]), get_truth(e)[f])[0][2]
        for e in val_data
    )
    print(f"{f}: lookup vocabulary = {len(vocab)} names from train labels")

F = lambda r, f: r["summary"]["fields"][f]["f1"]
print(
    f"\n{'field':20s} {'v3':>7s} {'base':>7s} {'all-null':>9s} {'lookup':>8s}   (mean F1, all val rows)"
)
for f in FIELDS:
    lk = f"{lookup[f]:8.3f}" if f in lookup else f"{'-':>8s}"
    print(f"{f:20s} {F(res, f):7.3f} {F(base_res, f):7.3f} {allnull[f]:9.3f} {lk}")
print(
    "\nReference values from the plan (hard-coded earlier, source unknown): metrics 0.847, datasets 0.612"
)

metrics: lookup vocabulary = 32 names from train labels
datasets: lookup vocabulary = 19 names from train labels

field                     v3    base  all-null   lookup   (mean F1, all val rows)
research_question      0.996   0.294     0.000        -
method                 0.992   0.077     0.768        -
datasets               0.943   0.205     0.751    0.832
metrics                0.963   0.325     0.584    0.855
key_finding            0.686   0.084     0.601        -

Reference values from the plan (hard-coded earlier, source unknown): metrics 0.847, datasets 0.612


# 11: Span Fields: Null Decision vs. Span Quality

In [11]:
def span_breakdown(r, label):
    for f in ("method", "key_finding"):
        nn = [e for e in r["examples"] if e["predicted"] and not is_null(e["ground_truth"][f])]
        nl = [e for e in r["examples"] if e["predicted"] and is_null(e["ground_truth"][f])]
        f1_nn = (
            mean(score_field(f, e["predicted"].get(f), e["ground_truth"][f])[0][2] for e in nn)
            if nn
            else float("nan")
        )
        null_ok = mean(is_null(e["predicted"].get(f)) for e in nl) if nl else float("nan")
        print(
            f"[{label:4s}] {f:12s} truth non-null (n={len(nn)}): mean F1 {f1_nn:.3f} | "
            f"truth null (n={len(nl)}): predicts null {null_ok:.0%}"
        )


span_breakdown(res, "v3")
span_breakdown(base_res, "base")

[v3  ] method       truth non-null (n=53): mean F1 0.983 | truth null (n=179): predicts null 100%
[v3  ] key_finding  truth non-null (n=93): mean F1 0.385 | truth null (n=139): predicts null 89%
[base] method       truth non-null (n=46): mean F1 0.390 | truth null (n=176): predicts null 0%
[base] key_finding  truth non-null (n=92): mean F1 0.202 | truth null (n=130): predicts null 1%


# 12: Gold Set Evaluation (method and key_finding)

In [12]:
gold_scores = None
if gold is None:
    print("No gold file found in /kaggle/input: skipping.")
else:

    def gold_eval(r):
        out = {}
        for f, gk in (("method", "gold_method"), ("key_finding", "gold_key_finding")):
            f1s, pred_null = [], 0
            for ex in r["examples"]:
                g = gold.get(val_data[ex["index"]]["metadata"]["chunk_id"])
                if not g or is_null(g.get(gk)):
                    continue
                pred = (ex["predicted"] or {}).get(f)
                pred_null += is_null(pred)
                f1s.append(score_field(f, pred, g[gk])[0][2])  # unparseable output counts as 0
            out[f] = {"n": len(f1s), "f1": mean(f1s) if f1s else None, "predicted_null": pred_null}
        return out

    gold_scores = {"v3": gold_eval(res), "base": gold_eval(base_res)}
    n_in_val = sum(g in va_chunks for g in gold)
    print(f"gold chunks present in val: {n_in_val} of {len(gold)}")
    for f in ("method", "key_finding"):
        for name in ("v3", "base"):
            s = gold_scores[name][f]
            f1 = "n/a" if s["f1"] is None else f"{s['f1']:.3f}"
            print(
                f"[{name:4s}] gold {f:12s} n={s['n']:3d}  F1={f1}  predicted null on {s['predicted_null']} of them"
            )
    if gold_scores["v3"]["method"]["n"] == 0 and gold_scores["v3"]["key_finding"]["n"] == 0:
        print(
            "\nWARNING: no gold row matched a val chunk, so there are no gold scores. "
            "Check that the gold chunk_ids belong to the val set."
        )

gold chunks present in val: 20 of 20
[v3  ] gold method       n=  5  F1=0.441  predicted null on 0 of them
[base] gold method       n=  5  F1=0.265  predicted null on 2 of them
[v3  ] gold key_finding  n= 10  F1=0.258  predicted null on 0 of them
[base] gold key_finding  n= 10  F1=0.544  predicted null on 0 of them


# 13: Success Criteria for v3 (Pass/Fail) and Save Summary

In [13]:
checks = []


def chk(name, ok, detail):
    checks.append((name, "PASS" if ok else "FAIL", detail))


v3s = res["summary"]
chk("JSON validity > 95%", v3s["json_valid_rate"] > 0.95, f"{v3s['json_valid_rate']:.1%}")
chk(
    "method F1 > 0.80 and > base",
    F(res, "method") > 0.80 and F(res, "method") > F(base_res, "method"),
    f"v3 {F(res, 'method'):.3f} | base {F(base_res, 'method'):.3f}",
)
chk(
    "datasets F1 > lookup and > base",
    F(res, "datasets") > lookup["datasets"] and F(res, "datasets") > F(base_res, "datasets"),
    f"v3 {F(res, 'datasets'):.3f} | lookup {lookup['datasets']:.3f} | base {F(base_res, 'datasets'):.3f}",
)
chk(
    "key_finding F1 > 0.60, > base and > all-null",
    F(res, "key_finding") > 0.60
    and F(res, "key_finding") > F(base_res, "key_finding")
    and F(res, "key_finding") > allnull["key_finding"],
    f"v3 {F(res, 'key_finding'):.3f} | base {F(base_res, 'key_finding'):.3f} | all-null {allnull['key_finding']:.3f}",
)
chk(
    "metrics F1 > lookup and > base",
    F(res, "metrics") > lookup["metrics"] and F(res, "metrics") > F(base_res, "metrics"),
    f"v3 {F(res, 'metrics'):.3f} | lookup {lookup['metrics']:.3f} | base {F(base_res, 'metrics'):.3f}",
)

for f, thr in (("method", 0.70), ("key_finding", 0.50)):
    s = gold_scores["v3"][f] if gold_scores else None
    if s and s["n"] and s["f1"] is not None:
        chk(f"gold {f} F1 > {thr:.2f}", s["f1"] > thr, f"{s['f1']:.3f} on n={s['n']}")
    else:
        checks.append((f"gold {f} F1 > {thr:.2f}", "SKIPPED", "no gold scores available"))

print(f"{'criterion':46s} {'result':8s} detail")
for name, status, detail in checks:
    print(f"{name:46s} {status:8s} {detail}")

summary = {
    "v3": res["summary"],
    "base": base_res["summary"],
    "all_null_f1": allnull,
    "lookup_f1": lookup,
    "gold": gold_scores,
    "criteria": [{"name": n, "status": s, "detail": d} for n, s, d in checks],
}
with open("/kaggle/working/extraction_eval_v3_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print("\nSaved -> /kaggle/working/extraction_eval_v3_summary.json")

criterion                                      result   detail
JSON validity > 95%                            PASS     99.6%
method F1 > 0.80 and > base                    PASS     v3 0.992 | base 0.077
datasets F1 > lookup and > base                PASS     v3 0.943 | lookup 0.832 | base 0.205
key_finding F1 > 0.60, > base and > all-null   PASS     v3 0.686 | base 0.084 | all-null 0.601
metrics F1 > lookup and > base                 PASS     v3 0.963 | lookup 0.855 | base 0.325
gold method F1 > 0.70                          FAIL     0.441 on n=5
gold key_finding F1 > 0.50                     FAIL     0.258 on n=10

Saved -> /kaggle/working/extraction_eval_v3_summary.json


# 14: Check the Gold Disagreement (No GPU)

In [14]:
# 1) Is the paper title now part of the model input?
users = [split_messages(e)[1].lower() for e in val_data]
hits = n = 0
for i, t in enumerate(va_t):
    m = re.search(r"the paper '(.*)'", t["research_question"])
    if m:
        n += 1
        hits += m.group(1).lower() in users[i]
print(f"research_question: title appears in the model input for {hits}/{n} val rows\n")


# 2) Do the normal labels agree with the gold rows?
def gold_of(ex):
    return gold.get(val_data[ex["index"]]["metadata"]["chunk_id"])


for field, gk in (("method", "gold_method"), ("key_finding", "gold_key_finding")):
    sc = [
        score_field(field, ex["ground_truth"][field], gold_of(ex)[gk])[0][2]
        for ex in res["examples"]
        if gold_of(ex) and not is_null(gold_of(ex).get(gk))
    ]
    print(f"{field}: normal label vs gold F1 = {mean(sc):.3f} (n={len(sc)})")


# 3) Side by side on the gold rows
def show(field, gk, limit=15):
    k = 0
    for ev, eb in zip(res["examples"], base_res["examples"], strict=False):
        g = gold_of(ev)
        if not g or is_null(g.get(gk)):
            continue
        pv = (ev["predicted"] or {}).get(field)
        pb = (eb["predicted"] or {}).get(field)
        print(
            f"\n[{ev['index']}] {field}\n  GOLD : {str(g[gk])[:300]}\n  LABEL: {str(ev['ground_truth'][field])[:300]}"
            f"\n  v3   : {str(pv)[:300]}\n  base : {str(pb)[:300]}"
        )
        k += 1
        if k >= limit:
            break


print()
show("method", "gold_method")
show("key_finding", "gold_key_finding")

research_question: title appears in the model input for 227/233 val rows

method: normal label vs gold F1 = 0.448 (n=5)
key_finding: normal label vs gold F1 = 0.286 (n=10)


[9] method
  GOLD : This paper addresses this shortcoming using a novel approach that applies natural language processing techniques to countries' annual statements in the UN General Debate.
  LABEL: of composition in STM allow us to pick up statistical uncertainty in the linear regression model.", 'Figure FIGREF9 demonstrates the effect of wealth (GDP per capita) on the the extent to which states discuss the two international development topics in their GD statements. The figure shows that the 
  v3   : of composition in STM allow us to pick up statistical uncertainty in the linear regression model.", 'Figure FIGREF9 demonstrates the effect of wealth (GDP per capita) on the the extent to which states discuss the two international development topics in their GD statements. The figure shows that the 
  base : None

